# Day 1: Quantitative Foundations & Mathematical Verification
**Prepared by:** Aditya (Lead), Anhum (Data), Tanveer (EDA & Verification)

This interactive notebook demonstrates and verifies the fundamental mathematical relationships in financial data science:
1. Arithmetic vs. Logarithmic returns
2. Intertemporal vs. cross-sectional additivity
3. Compounding wealth index trajectories
4. Configuration loading via `ConfigParser`

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config_parser import ConfigParser
from src.utils.finance_math import (
    calculate_arithmetic_returns,
    calculate_log_returns,
    log_to_arithmetic,
    compound_returns,
    annualize_return,
    annualize_volatility,
)

print("Environment ready and modules loaded successfully.")

## 1. Load System Configuration

In [ ]:
cfg = ConfigParser("../config/config.yaml")
print("Project:", cfg.get("project", "name"))
print("Equities:", cfg.get("assets", "equities"))
print("Initial Capital: $", f"{cfg.get('backtest', 'initial_capital'):,.2f}")

## 2. Mathematical Comparison: Arithmetic vs. Log Returns
Notice how for small returns, $r_t \approx R_t$. However, during large drawdowns or rallies, they diverge significantly.

In [ ]:
# Simulated realistic price trajectory
np.random.seed(42)
daily_shocks = np.random.normal(loc=0.0005, scale=0.015, size=252)
simulated_prices = pd.Series(100.0 * np.cumprod(1.0 + daily_shocks), name="Price")

r_arith = calculate_arithmetic_returns(simulated_prices)
r_log = calculate_log_returns(simulated_prices)

comparison_df = pd.DataFrame({
    "Price": simulated_prices.iloc[1:],
    "Arithmetic_Return": r_arith,
    "Log_Return": r_log,
    "Difference (Arith - Log)": r_arith - r_log
})

comparison_df.head(10)

## 3. Visualizing Cumulative Wealth Trajectory

In [ ]:
wealth_index = compound_returns(r_arith)

plt.figure(figsize=(10, 4))
plt.plot(wealth_index.values, label="Wealth Index ($1 Initial)", color="#1f77b4", lw=1.5)
plt.axhline(1.0, color="gray", linestyle="--", alpha=0.7)
plt.title("Cumulative Wealth Trajectory (252 Trading Days)")
plt.xlabel("Trading Day")
plt.ylabel("Portfolio Value ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()